# ARRGO: Theoretical Foundations

This notebook establishes the mathematical assumptions, properties, and
theoretical conditions required to analyze the correctness, convergence,
and optimality guarantees of ARRGO.

## Theoretical Assumptions

To analyze the correctness, convergence, and optimality properties of ARRGO,
we explicitly state the assumptions under which the theoretical results hold.

The assumptions are divided into two categories:

1. **Baseline assumptions**, which define the optimization problem and the
   validity of the refinement process.
2. **Certified-mode assumptions**, which are required when ARRGO uses rigorous
   uncertainty bounds and optimality certificates.

These assumptions describe the mathematical setting of the analysis. They do
not necessarily represent information that must be explicitly provided to the
algorithm during every run.

### Baseline Assumptions

#### 1. Bounded Search Domain

The optimization domain is a non-empty compact interval

$$
\Omega=[a,b],
\qquad a<b.
$$

Therefore, the search space is bounded and has finite diameter

$$
\operatorname{diam}(\Omega)=b-a.
$$

#### 2. Objective Function Evaluability

The objective function is deterministic and can be evaluated at any valid
point in the search domain:

$$
x\in\Omega
\quad\Longrightarrow\quad
f(x)\in\mathbb{R}.
$$

ARRGO does not require the analytical expression of $f$ to be available.

#### 3. Continuity of the Objective Function

For the theoretical analysis of global optimum existence, the objective
function is assumed to be continuous on $\Omega$:

$$
f\in C(\Omega).
$$

Since $\Omega$ is compact and $f$ is continuous, a global maximum exists:

$$
f^*=\max_{x\in\Omega} f(x).
$$

Thus, the global optimization problem is well-defined.

#### 4. Valid Refinement

Every refinement operation must preserve the validity of the search domain.

For a split of a region

$$
R=[l,r]
$$

at a point $s\in(l,r)$, the resulting regions are

$$
R_L=[l,s],
\qquad
R_R=[s,r].
$$

They must satisfy

$$
R_L\cup R_R=R.
$$

Furthermore, every accepted split satisfies the contraction condition

$$
\max(s-l,r-s)\leq\rho(r-l),
\qquad 0<\rho<1.
$$

#### 5. Valid Function Evaluations

Every function evaluation performed by ARRGO corresponds to a point inside the
search domain:

$$
x_{\mathrm{new}}\in\Omega.
$$

Duplicate evaluations are avoided up to the numerical spatial tolerance
defined by the implementation.

#### 6. Global Selection Fairness

The global region-selection mechanism must not permanently ignore a region that
remains relevant to the optimization process.

In particular, if a region remains capable of containing information that can
affect the global optimization decision, the selection mechanism must allow
that region to be selected for further refinement.

This condition will be formalized later as part of the convergence analysis.

### Certified-Mode Assumptions

The following assumptions are required only when ARRGO operates in certified
mode.

#### 7. Known Valid Lipschitz Bound

The objective function is assumed to satisfy a globally valid Lipschitz
condition with constant $L>0$:

$$
|f(x)-f(y)|
\leq
L|x-y|,
\qquad
\forall x,y\in\Omega.
$$

The constant $L$ must be a valid upper bound on the true Lipschitz constant of
the objective function.

Under this assumption, ARRGO can construct rigorous lower and upper
function-value bounds from observed function evaluations.

#### 8. Deterministic Exact-Evaluation Model

For the baseline theoretical results, function evaluations are treated as
exact:

$$
y_i=f(x_i).
$$

Measurement noise and stochastic objective functions are outside the scope of
the current theoretical analysis.

### Separation of Assumptions and Guarantees

Not every assumption is required for every ARRGO capability.

In particular:

- Continuity is used to establish the existence of a global optimum on the
  compact domain.
- Contraction is used to establish spatial refinement.
- Global selection fairness is required for the convergence analysis.
- A valid Lipschitz bound is required for rigorous uncertainty bounds and
  certified optimality guarantees.
- Exact evaluations are assumed for the idealized deterministic theoretical
  model.

Therefore, ARRGO distinguishes between **algorithmic behavior** and
**theoretical guarantees**.

$$
\boxed{
\text{Guarantee}
\;\Longrightarrow\;
\text{Assumptions}
+
\text{Valid Refinement}
+
\text{Valid Analysis}
}
$$

## Existence of a Global Optimum

The existence of a global optimum is a fundamental requirement for the
theoretical analysis of ARRGO.

Under the baseline assumptions, the search domain $\Omega$ is compact and the
objective function $f$ is continuous on $\Omega$:

$$
\Omega=[a,b],
\qquad
f\in C(\Omega).
$$

By the Extreme Value Theorem (Weierstrass theorem), a continuous function on a
compact domain attains both its maximum and minimum.

Therefore, there exists at least one point

$$
x^*\in\Omega
$$

such that

$$
f(x^*)=\max_{x\in\Omega}f(x).
$$

We define the global optimal value as

$$
f^*=f(x^*)
=
\max_{x\in\Omega}f(x).
$$

The set of global maximizers is

$$
X^*
=
\operatorname*{arg\,max}_{x\in\Omega}f(x).
$$

In general, $X^*$ may contain more than one point. ARRGO therefore does not
assume that the global maximizer is unique.

### Why This Matters for ARRGO

The existence result gives ARRGO a well-defined target:

$$
\boxed{
\text{Global optimization target}
=
\max_{x\in\Omega}f(x)
}
$$

This distinction is important because the algorithm may first obtain a
**best-found solution**

$$
f_{\mathrm{best}}
=
\max_{x_i\in\mathcal D}f(x_i),
$$

while the true global optimum is

$$
f^*
=
\max_{x\in\Omega}f(x).
$$

Since the evaluated set $\mathcal D$ is generally only a finite subset of
$\Omega$,

$$
f_{\mathrm{best}}\leq f^*.
$$

Equality holds only when the evaluated information is sufficient to identify
a globally optimal value.

Thus, the theoretical analysis must distinguish between:

$$
\boxed{
\text{Best Found Solution}
\neq
\text{True Global Optimum}
}
$$

unless additional conditions provide a valid optimality certificate.

## Continuity and Compactness

Continuity and compactness provide the mathematical foundation for the
existence and stability of the global optimization problem considered by
ARRGO.

### Continuity

The objective function is assumed to be continuous on the search domain:

$$
f\in C(\Omega).
$$

Continuity means that sufficiently small changes in the input produce
sufficiently small changes in the objective value.

Formally, for every $\varepsilon>0$ and every $x\in\Omega$, there exists
$\delta>0$ such that

$$
|x-y|<\delta
\quad\Longrightarrow\quad
|f(x)-f(y)|<\varepsilon.
$$

This property is important for adaptive region refinement because ARRGO
progressively reduces the size of regions.

If a sequence of points satisfies

$$
x_k\to x,
$$

then continuity guarantees

$$
f(x_k)\to f(x).
$$

Therefore, as the spatial resolution of a region increases, the function
values observed within that region become increasingly related to its local
behavior.

### Compactness

The search domain is assumed to be the closed and bounded interval

$$
\Omega=[a,b],
\qquad a<b.
$$

In $\mathbb{R}$, every closed and bounded interval is compact.

Compactness provides two important properties for ARRGO.

First, together with continuity, it guarantees the existence of a global
maximum:

$$
\exists x^*\in\Omega:
\qquad
f(x^*)=\max_{x\in\Omega}f(x).
$$

Second, compactness ensures that the entire optimization problem is contained
inside a finite spatial domain:

$$
\operatorname{diam}(\Omega)=b-a<\infty.
$$

This finite diameter allows ARRGO to reason about spatial refinement through
the sizes of regions.

### Interaction with Region Refinement

Consider a sequence of nested regions

$$
R_0\supseteq R_1\supseteq R_2\supseteq\cdots
$$

generated by ARRGO.

If the refinement mechanism satisfies the contraction condition

$$
\operatorname{diam}(R_{k+1})
\leq
\rho\,\operatorname{diam}(R_k),
\qquad
0<\rho<1,
$$

then

$$
\operatorname{diam}(R_k)
\leq
\rho^k\operatorname{diam}(R_0).
$$

Consequently,

$$
\lim_{k\to\infty}\operatorname{diam}(R_k)=0.
$$

Continuity then implies that function values within sufficiently refined
regions become increasingly close whenever their spatial distance becomes
sufficiently small.

This provides the basic connection between ARRGO's spatial refinement and
the mathematical behavior of the objective function:

$$
\boxed{
\text{Compact Domain}
\;+\;
\text{Continuity}
\;+\;
\text{Spatial Contraction}
}
$$

creates the foundation for analyzing increasingly localized regions around
potential optima.

However, continuity and contraction alone do not provide a numerical bound on
the unknown objective values inside a region. Rigorous numerical bounds
require additional assumptions, such as the Lipschitz condition introduced
later.

## Lipschitz Regularity

Continuity guarantees that small changes in the input produce small changes
in the objective value, but it does not provide a quantitative bound on the
magnitude of those changes.

For certified uncertainty estimation, ARRGO may additionally assume that the
objective function satisfies a global Lipschitz condition.

### Lipschitz Condition

A function $f:\Omega\rightarrow\mathbb{R}$ is Lipschitz continuous on $\Omega$
if there exists a constant $L\geq0$ such that

$$
|f(x)-f(y)|
\leq
L|x-y|,
\qquad
\forall x,y\in\Omega.
$$

The constant $L$ is called a Lipschitz constant of $f$.

A valid Lipschitz constant does not need to be the smallest possible constant.
Any value satisfying the inequality for every pair of points in $\Omega$ is
sufficient for the theoretical guarantees.

### Interpretation

The Lipschitz condition places a quantitative limit on how rapidly the
objective function can change with respect to the input.

For two points $x$ and $y$,

$$
|f(x)-f(y)|
\leq
L|x-y|.
$$

Therefore, if the distance between two points is small, the maximum possible
difference between their function values is also bounded.

For example, if an evaluated point $x_i$ has value

$$
y_i=f(x_i),
$$

then for any other point $x\in\Omega$,

$$
|f(x)-y_i|
\leq
L|x-x_i|.
$$

Equivalently,

$$
y_i-L|x-x_i|
\leq
f(x)
\leq
y_i+L|x-x_i|.
$$

These inequalities provide lower and upper bounds on the unknown value
$f(x)$.

### Relation to Region Refinement

Consider a region

$$
R=[l,r].
$$

For any two points $x,y\in R$,

$$
|x-y|
\leq
\operatorname{diam}(R)
=
r-l.
$$

Therefore,

$$
|f(x)-f(y)|
\leq
L(r-l).
$$

As ARRGO refines a region and its diameter decreases, the maximum possible
variation implied by the Lipschitz condition also decreases.

If

$$
\operatorname{diam}(R_k)\to0,
$$

then

$$
L\operatorname{diam}(R_k)\to0.
$$

Thus, spatial contraction directly reduces the maximum function-value
variation allowed by the Lipschitz model.

### Role in Certified ARRGO

The Lipschitz assumption enables ARRGO to transform observed function
evaluations into rigorous bounds over unobserved points.

This allows the algorithm to distinguish between two different concepts:

$$
\boxed{
\text{Observed Information}
}
$$

and

$$
\boxed{
\text{Certified Information About Unobserved Points}
}
$$

The former is available from function evaluations alone, while the latter
requires a valid mathematical assumption such as the Lipschitz condition.

Therefore, Lipschitz continuity is **not required for the basic operation of
ARRGO**. It is an additional assumption used by the certified version of the
framework.

### Important Distinction

The Lipschitz constant used by ARRGO must be a valid upper bound.

An estimated value $\widehat L$ is not automatically a valid Lipschitz
constant:

$$
\widehat L
\not\Rightarrow
\text{Certified Bound}.
$$

If

$$
\widehat L < L_{\mathrm{true}},
$$

the resulting envelopes may fail to contain the true function and therefore
cannot provide a rigorous optimality certificate.

Consequently,

$$
\boxed{
\text{Certified ARRGO}
\Longrightarrow
\text{Valid Lipschitz Bound}
}
$$

while

$$
\boxed{
\text{Basic ARRGO}
\not\Longrightarrow
\text{Lipschitz Assumption}
}
$$

## Validity of Lipschitz-Based Bounds

Assume that $f$ satisfies the Lipschitz condition

$$
|f(x)-f(y)|
\leq
L|x-y|,
\qquad
\forall x,y\in\Omega.
$$

Suppose ARRGO has evaluated the objective function at the points

$$
\mathcal D_R
=
\{(x_i,y_i)\}_{i=1}^{n},
\qquad
y_i=f(x_i).
$$

For any point $x\in R$, applying the Lipschitz condition to $x$ and each
observed point $x_i$ gives

$$
|f(x)-y_i|
\leq
L|x-x_i|.
$$

Therefore,

$$
-L|x-x_i|
\leq
f(x)-y_i
\leq
L|x-x_i|,
$$

which is equivalent to

$$
y_i-L|x-x_i|
\leq
f(x)
\leq
y_i+L|x-x_i|.
$$

Thus, every observed point provides an independent lower and upper bound on
the unknown value $f(x)$.

### Lower Envelope

Since the lower bound must hold for every observed point,

$$
f(x)
\geq
y_i-L|x-x_i|,
\qquad
i=1,\ldots,n.
$$

Therefore, the strongest lower bound obtained from all observations is

$$
\boxed{
L_R(x)
=
\max_{1\leq i\leq n}
\left[
y_i-L|x-x_i|
\right]
}
$$

and consequently,

$$
L_R(x)\leq f(x).
$$

### Upper Envelope

Similarly, every observed point provides an upper bound

$$
f(x)
\leq
y_i+L|x-x_i|.
$$

The strongest upper bound consistent with all observations is therefore

$$
\boxed{
U_R(x)
=
\min_{1\leq i\leq n}
\left[
y_i+L|x-x_i|
\right]
}
$$

and consequently,

$$
f(x)\leq U_R(x).
$$

### Validity Theorem

Combining the two inequalities gives

$$
\boxed{
L_R(x)
\leq
f(x)
\leq
U_R(x),
\qquad
\forall x\in R.
}
$$

Therefore, the interval

$$
[L_R(x),U_R(x)]
$$

is a valid pointwise enclosure of the objective value at $x$.

### Proof

For every observation $(x_i,y_i)$,

$$
y_i-L|x-x_i|
\leq
f(x).
$$

Since this inequality holds for all $i$, it also holds for their maximum:

$$
\max_i
\left[
y_i-L|x-x_i|
\right]
\leq
f(x).
$$

Hence,

$$
L_R(x)\leq f(x).
$$

Likewise, for every observation,

$$
f(x)
\leq
y_i+L|x-x_i|.
$$

Since this inequality holds for all $i$, it also holds for their minimum:

$$
f(x)
\leq
\min_i
\left[
y_i+L|x-x_i|
\right].
$$

Hence,

$$
f(x)\leq U_R(x).
$$

Combining both results,

$$
L_R(x)
\leq
f(x)
\leq
U_R(x).
$$

Therefore, the Lipschitz-based envelopes are valid bounds on the objective
function over the region.

### Consequence for ARRGO

The validity of these envelopes means that ARRGO can reason about points that
have never been evaluated directly.

The uncertainty at a point can be defined as

$$
u_R(x)
=
U_R(x)-L_R(x).
$$

Because the true function value is enclosed by the two bounds,

$$
f(x)\in[L_R(x),U_R(x)].
$$

Thus, the uncertainty measure is not merely an empirical estimate when the
Lipschitz assumption is valid. It represents a mathematically justified
interval of possible function values.

This establishes the theoretical foundation for ARRGO's certified uncertainty
and optimization-potential mechanisms.

## Spatial Contraction

Spatial contraction is the structural mechanism that guarantees that repeated
region refinement produces increasingly smaller search regions.

Consider a region

$$
R=[l,r]
$$

and a split point

$$
s\in(l,r).
$$

The split produces two child regions

$$
R_L=[l,s],
\qquad
R_R=[s,r].
$$

The contraction condition requires

$$
\max(s-l,r-s)
\leq
\rho(r-l),
\qquad
0<\rho<1.
$$

This condition guarantees that the diameter of every child region is strictly
smaller than the diameter of its parent.

### One-Step Contraction

The diameter of the parent region is

$$
\operatorname{diam}(R)=r-l.
$$

The diameters of the two child regions are

$$
\operatorname{diam}(R_L)=s-l
$$

and

$$
\operatorname{diam}(R_R)=r-s.
$$

By the contraction condition,

$$
\max
\left\{
\operatorname{diam}(R_L),
\operatorname{diam}(R_R)
\right\}
\leq
\rho\operatorname{diam}(R).
$$

Since

$$
0<\rho<1,
$$

we have

$$
\rho\operatorname{diam}(R)
<
\operatorname{diam}(R).
$$

Therefore,

$$
\boxed{
\operatorname{diam}(R_L)<\operatorname{diam}(R)
}
$$

and

$$
\boxed{
\operatorname{diam}(R_R)<\operatorname{diam}(R).
}
$$

Thus, every accepted split produces strictly smaller child regions.

### Repeated Contraction

Consider a sequence of regions generated by repeated refinement:

$$
R_0\supseteq R_1\supseteq R_2\supseteq\cdots
$$

where each refinement satisfies the contraction condition.

Then

$$
\operatorname{diam}(R_{k+1})
\leq
\rho\operatorname{diam}(R_k).
$$

Applying this relation repeatedly gives

$$
\operatorname{diam}(R_k)
\leq
\rho^k\operatorname{diam}(R_0).
$$

Because

$$
0<\rho<1,
$$

we have

$$
\lim_{k\to\infty}\rho^k=0.
$$

Consequently,

$$
\boxed{
\lim_{k\to\infty}
\operatorname{diam}(R_k)
=
0.
}
$$

Therefore, any infinite sequence of refinements along a nested path produces
regions whose spatial diameter converges to zero.

### Relation to Function Resolution

Under the Lipschitz assumption,

$$
|f(x)-f(y)|
\leq
L|x-y|.
$$

For any two points $x,y\in R_k$,

$$
|x-y|
\leq
\operatorname{diam}(R_k).
$$

Therefore,

$$
|f(x)-f(y)|
\leq
L\operatorname{diam}(R_k).
$$

Since

$$
\operatorname{diam}(R_k)\to0,
$$

it follows that

$$
L\operatorname{diam}(R_k)\to0.
$$

Hence, the maximum variation allowed by the Lipschitz condition inside the
region also converges to zero.

This establishes the following relationship:

$$
\boxed{
\text{Spatial Contraction}
\Longrightarrow
\text{Decreasing Spatial Uncertainty}
}
$$

and, under a valid Lipschitz bound,

$$
\boxed{
\text{Spatial Contraction}
\Longrightarrow
\text{Decreasing Function-Value Variation}
}
$$

### Role in ARRGO Convergence

Spatial contraction alone does not prove that ARRGO finds the global optimum.

It provides the structural component required for convergence analysis:

$$
\text{Repeated Refinement}
\Longrightarrow
\text{Shrinking Regions}.
$$

To obtain a global optimization guarantee, this property must be combined with
a valid information mechanism and a global region-selection condition.

Therefore, the convergence argument will later rely on the combination

$$
\boxed{
\text{Spatial Contraction}
+
\text{Information Refinement}
+
\text{Global Selection}
}
$$

rather than on contraction alone.

## Information Resolution

ARRGO does not refine regions only to reduce their spatial size. The purpose of
spatial refinement is to obtain increasingly informative observations about the
objective function.

Let a region be represented by

$$
R=(I,\mathcal D_R),
$$

where $I$ is its spatial domain and $\mathcal D_R$ is the set of available
function evaluations inside or relevant to the region.

The information state of a region can be represented abstractly as

$$
S_R=(\mathcal D_R,B_R,\mathcal Q_R),
$$

where:

- $\mathcal D_R$ represents observed function values,
- $B_R$ represents the observed behavioral profile,
- $\mathcal Q_R$ represents unresolved information.

### Spatial Resolution

The spatial resolution of a region is related to its diameter:

$$
\operatorname{diam}(R).
$$

Under the contraction condition,

$$
\operatorname{diam}(R_k)
\leq
\rho^k\operatorname{diam}(R_0),
\qquad
0<\rho<1.
$$

Therefore,

$$
\operatorname{diam}(R_k)\to0.
$$

A smaller region provides a more localized spatial context for interpreting
function evaluations.

### Information Resolution

Let

$$
\mathcal Q_R
=
[
Q_{\mathrm{coverage}},
Q_{\mathrm{behavior}},
Q_{\mathrm{uncertainty}},
Q_{\mathrm{potential}}
]
$$

denote the unresolved information profile of a region.

Refinement is considered successful when it reduces the information relevant
to the current optimization decision.

For a refinement action $A$, let the resulting information state be

$$
S_R^A.
$$

The corresponding unresolved information profile is

$$
\mathcal Q_R^A.
$$

The conceptual information improvement produced by the action is therefore

$$
\Delta\mathcal Q(A\mid R)
=
\mathcal Q_R-\mathcal Q_R^A.
$$

This expression represents a reduction in unresolved information rather than a
single numerical objective.

### Sampling and Information Resolution

A sampling operation adds a new observation:

$$
\mathcal D_R'
=
\mathcal D_R
\cup
\{(x_{\mathrm{new}},f(x_{\mathrm{new}}))\}.
$$

The new observation may reduce uncertainty about:

- local function behavior,
- directional changes,
- candidate extrema,
- spatial coverage,
- optimization potential.

Therefore,

$$
\boxed{
\text{Sampling}
\Longrightarrow
\text{Information Refinement}
}
$$

### Splitting and Information Resolution

A splitting operation changes the spatial structure of the problem:

$$
R\rightarrow\{R_L,R_R\}.
$$

The observations associated with the parent region can then be interpreted
within smaller spatial contexts.

This may reveal differences between subregions that were not sufficiently
visible at the parent scale.

Therefore,

$$
\boxed{
\text{Splitting}
\Longrightarrow
\text{Structural Information Refinement}
}
$$

### Information Resolution and Lipschitz Uncertainty

In certified mode, the Lipschitz condition provides

$$
|f(x)-f(y)|
\leq
L|x-y|.
$$

For points inside a region $R$,

$$
|f(x)-f(y)|
\leq
L\operatorname{diam}(R).
$$

Thus, as the region contracts,

$$
\operatorname{diam}(R)\to0
$$

and consequently,

$$
L\operatorname{diam}(R)\to0.
$$

This means that the maximum function-value variation permitted solely by the
regional diameter also decreases.

Therefore, spatial refinement contributes directly to information resolution
under the Lipschitz assumption.

### Important Limitation

Spatial contraction does not automatically imply that every aspect of the
objective function has been sufficiently observed.

A region can be spatially small while still containing unresolved information
if, for example, its observations are insufficient to characterize the local
behavior relevant to the optimization decision.

Therefore,

$$
\boxed{
\text{Small Region}
\neq
\text{Automatically Sufficient Information}
}
$$

ARRGO must evaluate both spatial resolution and information resolution.

The theoretical objective is therefore not simply

$$
\operatorname{diam}(R)\to0,
$$

but rather

$$
\boxed{
\text{Spatial Resolution}
+
\text{Information Resolution}
}
$$

as the foundation for subsequent convergence and optimality analysis.

## Optimization Potential Bound

The purpose of an optimization-potential bound is to quantify how good a
region could still be under the information currently available to ARRGO.

Consider a region

$$
R=[l,r]
$$

with a valid upper envelope

$$
f(x)\le U_R(x),
\qquad
\forall x\in R.
$$

Since the inequality holds for every point in the region, it also holds at a
global maximizer contained in that region.

### Regional Upper Potential

Define the optimization potential of the region as

$$
\boxed{
P(R)
=
\max_{x\in R} U_R(x)
}
$$

Because

$$
f(x)\le U_R(x),
\qquad
\forall x\in R,
$$

we have

$$
\max_{x\in R}f(x)
\leq
\max_{x\in R}U_R(x).
$$

Therefore,

$$
\boxed{
\max_{x\in R}f(x)\leq P(R)
}
$$

and $P(R)$ is a valid upper bound on the best objective value that can occur
inside region $R$.

### Relation to the Global Optimum

Let

$$
f^*
=
\max_{x\in\Omega}f(x)
$$

be the global optimal value.

If the global optimizer belongs to a region $R^*$, then

$$
f^*
=
\max_{x\in R^*}f(x).
$$

By the regional upper bound,

$$
f^*
\leq
P(R^*).
$$

Therefore, the potential of the region containing a global maximizer is always
at least as large as the true global optimum:

$$
\boxed{
f^*\leq P(R^*)
}
$$

This property makes regional potential useful for global optimization.

### Global Potential Bound

For a collection of regions

$$
\mathcal R=\{R_1,\ldots,R_m\},
$$

define the global potential as

$$
P_{\mathrm{global}}
=
\max_{R\in\mathcal R}P(R).
$$

Because the search domain is covered by the regions,

$$
\Omega
\subseteq
\bigcup_{R\in\mathcal R}R,
$$

the global optimum is contained in at least one represented region.

Consequently,

$$
\boxed{
f^*
\leq
P_{\mathrm{global}}
}
$$

and the global potential provides an upper bound on the unknown optimum.

### Relation to the Incumbent

ARRGO maintains the best function value obtained from actual evaluations:

$$
f_{\mathrm{best}}
=
\max_{x_i\in\mathcal D}f(x_i).
$$

Since every evaluated point belongs to the search domain,

$$
f_{\mathrm{best}}
\leq
f^*.
$$

Combining this with the global potential bound gives

$$
\boxed{
f_{\mathrm{best}}
\leq
f^*
\leq
P_{\mathrm{global}}
}
$$

This creates a mathematically meaningful interval containing the unknown global
optimal value.

### Global Optimality Gap

Define the certified global optimality gap as

$$
\boxed{
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}
}
$$

Then

$$
0
\leq
f^*-f_{\mathrm{best}}
\leq
\Delta_{\mathrm{global}}.
$$

Therefore, $\Delta_{\mathrm{global}}$ is a valid upper bound on the difference
between the best value found by ARRGO and the true global optimum.

### Epsilon-Optimality

If

$$
\Delta_{\mathrm{global}}
\leq
\varepsilon,
$$

then

$$
f^*-f_{\mathrm{best}}
\leq
\varepsilon.
$$

Hence,

$$
\boxed{
f_{\mathrm{best}}
\text{ is an }
\varepsilon\text{-optimal objective value}
}
$$

under the validity of the upper-bound construction.

This is stronger than simply reporting the best value observed during the
search.

### Certification Requirement

The optimality-gap guarantee depends critically on the validity of the upper
envelopes.

In particular,

$$
P_{\mathrm{global}}
$$

is a certified upper bound only when the assumptions used to construct
$U_R(x)$ are valid.

For the Lipschitz-based version of ARRGO, this requires a valid global
Lipschitz constant.

Therefore,

$$
\boxed{
\text{Valid Upper Bound}
\Longrightarrow
\text{Valid Optimality Gap}
}
$$

whereas an empirical or heuristic upper estimate does not by itself provide a
rigorous certificate.

### Interpretation for ARRGO

The optimization potential answers a different question from uncertainty.

Uncertainty asks:

$$
\boxed{
\text{How much is still unknown?}
}
$$

Optimization potential asks:

$$
\boxed{
\text{How good could this region still be?}
}
$$

ARRGO uses both concepts because a region may have high uncertainty without
being particularly relevant to the global optimum, or may have high
optimization potential and therefore require further refinement.

Thus,

$$
\boxed{
\text{Uncertainty}
\neq
\text{Optimization Potential}
}
$$

and both are required for a complete certified global optimization analysis.

## Global Optimality Gap

The global optimality gap measures the maximum possible difference between the
best objective value currently found by ARRGO and the true global optimum.

Let

$$
f_{\mathrm{best}}
=
\max_{x_i\in\mathcal D} f(x_i)
$$

denote the best value obtained from all evaluated points.

Let

$$
f^*
=
\max_{x\in\Omega}f(x)
$$

denote the true global optimal value.

Since the evaluated points are contained in the search domain,

$$
f_{\mathrm{best}}\leq f^*.
$$

In certified mode, ARRGO maintains a valid global upper bound

$$
P_{\mathrm{global}}
\geq
f^*.
$$

Therefore,

$$
\boxed{
f_{\mathrm{best}}
\leq
f^*
\leq
P_{\mathrm{global}}
}
$$

### Definition

The certified global optimality gap is defined as

$$
\boxed{
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}
}
$$

This quantity represents the largest remaining uncertainty about the
optimality of the incumbent that is justified by the current certified
information.

Because

$$
f_{\mathrm{best}}
\leq
f^*
\leq
P_{\mathrm{global}},
$$

we obtain

$$
0
\leq
f^*-f_{\mathrm{best}}
\leq
\Delta_{\mathrm{global}}.
$$

Thus, the actual error of the incumbent is never larger than the certified
gap.

### Interpretation

The gap can be interpreted as the maximum amount by which the current
incumbent could still be improved according to the available certified
information.

A large value of

$$
\Delta_{\mathrm{global}}
$$

means that the current information does not yet provide a strong guarantee
about global optimality.

A small value means that the remaining possible improvement is tightly
bounded.

Therefore,

$$
\boxed{
\Delta_{\mathrm{global}}
\rightarrow 0
}
$$

represents increasing certainty about the global optimality of the incumbent.

### Epsilon-Optimality

For a prescribed tolerance $\varepsilon>0$, if

$$
\Delta_{\mathrm{global}}
\leq
\varepsilon,
$$

then

$$
f^*-f_{\mathrm{best}}
\leq
\varepsilon.
$$

Equivalently,

$$
\boxed{
f_{\mathrm{best}}
\geq
f^*-\varepsilon
}
$$

and therefore the incumbent is $\varepsilon$-optimal in objective value.

This provides a quantitative stopping criterion that is directly connected to
the quality of the optimization result.

### Gap Reduction Through Refinement

Suppose ARRGO performs a refinement operation that produces new information
without invalidating the existing upper bounds.

Let the global upper bound before refinement be

$$
P_{\mathrm{global}}^{(t)}
$$

and after refinement be

$$
P_{\mathrm{global}}^{(t+1)}.
$$

If the new information tightens the certified upper bound, then

$$
P_{\mathrm{global}}^{(t+1)}
\leq
P_{\mathrm{global}}^{(t)}.
$$

At the same time, the incumbent cannot become worse:

$$
f_{\mathrm{best}}^{(t+1)}
\geq
f_{\mathrm{best}}^{(t)}.
$$

Therefore,

$$
\Delta_{\mathrm{global}}^{(t+1)}
=
P_{\mathrm{global}}^{(t+1)}
-
f_{\mathrm{best}}^{(t+1)}
\leq
\Delta_{\mathrm{global}}^{(t)}.
$$

Hence, under valid bound tightening,

$$
\boxed{
\Delta_{\mathrm{global}}
\text{ is non-increasing}
}
$$

through successful information refinement.

### Important Qualification

The monotonicity result depends on the upper bound being recomputed from
additional valid information.

It does not mean that every arbitrary sampling or splitting heuristic will
automatically reduce the gap.

ARRGO must therefore distinguish between:

$$
\boxed{
\text{Refinement}
}
$$

and

$$
\boxed{
\text{Effective Certified Refinement}
}
$$

The latter is refinement that preserves validity while improving the
information relevant to the global optimality bound.

### Relation to ARRGO Termination

The certified termination condition can be expressed as

$$
\boxed{
\Delta_{\mathrm{global}}\leq\varepsilon
}
$$

provided that all regional upper bounds contributing to
$P_{\mathrm{global}}$ are valid.

At that point, ARRGO does not merely return the best value observed. It can
return the incumbent together with a mathematical statement that its
objective value is within $\varepsilon$ of the true global optimum.

This distinction separates a certified optimization result from a
budget-limited best-found result.

## Epsilon-Optimality

A global optimization algorithm does not always need to identify the exact
global optimizer in order to provide a mathematically meaningful guarantee.

For a prescribed tolerance

$$
\varepsilon>0,
$$

an objective value is called $\varepsilon$-optimal if its distance from the
true global optimal value is no greater than $\varepsilon$.

### Definition

Let

$$
f^*
=
\max_{x\in\Omega}f(x)
$$

be the global optimal value, and let $\hat{x}\in\Omega$ be a candidate
solution.

The candidate $\hat{x}$ is $\varepsilon$-optimal in objective value if

$$
f^*-f(\hat{x})
\leq
\varepsilon.
$$

Equivalently,

$$
\boxed{
f(\hat{x})
\geq
f^*-\varepsilon
}
$$

This definition measures the quality of the objective value rather than the
distance between the candidate point and a particular global optimizer.

### Why Objective-Value Optimality Matters

A global optimization problem may have multiple global maximizers.

The set of global maximizers is

$$
X^*
=
\operatorname*{arg\,max}_{x\in\Omega}f(x).
$$

Therefore, requiring an algorithm to identify one specific optimizer is not
always necessary.

Moreover, two points can be far apart in the search domain while having
nearly identical objective values.

For this reason, ARRGO's primary certified accuracy measure is based on the
objective-value gap

$$
f^*-f(\hat{x}),
$$

rather than solely on the spatial distance

$$
|\hat{x}-x^*|.
$$

### ARRGO's Certified Condition

ARRGO maintains the incumbent

$$
x_{\mathrm{best}}
=
\operatorname*{arg\,max}_{x_i\in\mathcal D}f(x_i)
$$

with objective value

$$
f_{\mathrm{best}}
=
f(x_{\mathrm{best}}).
$$

In certified mode, suppose ARRGO maintains a valid global upper bound

$$
P_{\mathrm{global}}
\geq
f^*.
$$

The certified global optimality gap is

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}.
$$

Since

$$
f^*
\leq
P_{\mathrm{global}},
$$

we have

$$
f^*-f_{\mathrm{best}}
\leq
P_{\mathrm{global}}-f_{\mathrm{best}}
=
\Delta_{\mathrm{global}}.
$$

Therefore, if

$$
\boxed{
\Delta_{\mathrm{global}}\leq\varepsilon
}
$$

then

$$
f^*-f_{\mathrm{best}}
\leq
\varepsilon.
$$

Hence,

$$
\boxed{
x_{\mathrm{best}}
\text{ is an }\varepsilon\text{-optimal solution}
}
$$

in objective value.

### Certified Versus Best-Found Results

It is important to distinguish two possible termination situations.

#### Certified Termination

If

$$
\Delta_{\mathrm{global}}\leq\varepsilon,
$$

and the upper bounds are mathematically valid, ARRGO can provide the
certificate

$$
f_{\mathrm{best}}
\geq
f^*-\varepsilon.
$$

This is a theoretical guarantee.

#### Budget-Based Termination

If ARRGO reaches its evaluation budget before the certified gap satisfies the
desired tolerance,

$$
N_t=N_{\max},
$$

the algorithm returns the best evaluated solution:

$$
x_{\mathrm{best}}.
$$

However, without a sufficiently small certified gap, the algorithm cannot
claim that this solution is $\varepsilon$-optimal.

Thus,

$$
\boxed{
\text{Best Found}
\neq
\text{Certified }\varepsilon\text{-Optimal}
}
$$

in general.

### Exact Optimality as a Special Case

If the certified gap reaches zero,

$$
\Delta_{\mathrm{global}}=0,
$$

then

$$
f_{\mathrm{best}}
=
f^*.
$$

Therefore, the incumbent achieves the exact global optimal objective value.

In practice, numerical computation generally uses a positive tolerance
$\varepsilon$, making $\varepsilon$-optimality the more practical stopping
criterion.

### Role in ARRGO

Epsilon-optimality connects the information maintained by ARRGO to a
quantitative statement about solution quality:

$$
\boxed{
\text{Certified Information}
\Longrightarrow
\text{Optimality Gap}
\Longrightarrow
\varepsilon\text{-Optimality}
}
$$

This establishes the mathematical meaning of ARRGO's certified termination
criterion.

## Convergence Conditions

The convergence analysis of ARRGO requires conditions that connect local
region refinement to the global optimization problem.

The objective is to establish that, under appropriate assumptions, the
algorithm cannot permanently exclude a region containing a global optimizer
and that the spatial resolution of relevant regions eventually becomes
arbitrarily fine.

The following conditions are required for the convergence analysis.

### Condition 1: Compact Search Domain

The search domain is a non-empty compact interval

$$
\Omega=[a,b],
\qquad a<b.
$$

Compactness guarantees the existence of a global maximizer when combined with
continuity of the objective function.

### Condition 2: Continuity of the Objective Function

The objective function satisfies

$$
f\in C(\Omega).
$$

Therefore, if a sequence of points converges,

$$
x_k\to x,
$$

then

$$
f(x_k)\to f(x).
$$

This condition allows increasingly small regions around a global optimizer to
represent increasingly precise information about its objective value.

### Condition 3: Valid Region Coverage

At every iteration, the collection of active and stable regions must preserve
coverage of the original search domain:

$$
\Omega
\subseteq
\bigcup_{R\in\mathcal R_t}R.
$$

For a non-overlapping partition, this becomes

$$
\Omega
=
\bigcup_{R\in\mathcal R_t}R.
$$

Thus, refinement changes the representation of the search space without
removing any part of the original domain.

### Condition 4: Valid Spatial Refinement

Whenever a region

$$
R=[l,r]
$$

is structurally refined at a split point $s$, its children

$$
R_L=[l,s],
\qquad
R_R=[s,r]
$$

must satisfy

$$
\max\{s-l,r-s\}
\leq
\rho(r-l),
\qquad
0<\rho<1.
$$

Consequently, repeated refinement along a nested sequence of regions produces

$$
\operatorname{diam}(R_k)
\leq
\rho^k\operatorname{diam}(R_0)
\to 0.
$$

### Condition 5: Information Refinement

Spatial refinement alone is not sufficient for ARRGO's information-driven
decision process.

The algorithm must also continue acquiring or propagating relevant
information about regions that remain potentially important.

In particular, a region containing a global optimizer must not remain
permanently unresolved merely because other regions repeatedly receive
refinement.

This condition connects the local refinement mechanism to the global
selection mechanism.

### Condition 6: Global Selection Fairness

Let $\mathcal R_t^{\mathrm{relevant}}$ denote regions whose current
information does not allow them to be safely excluded from containing a
global optimizer.

The global selection mechanism must satisfy the fairness requirement that
such a region cannot be ignored indefinitely.

Informally,

$$
R\in\mathcal R_t^{\mathrm{relevant}}
\quad\Longrightarrow\quad
R\text{ receives refinement opportunities over time}.
$$

This does not require every region to be refined at every iteration.

Instead, it prevents permanent starvation of regions that remain globally
relevant.

### Condition 7: Monotonic Incumbent Improvement

The incumbent objective value is defined by

$$
f_{\mathrm{best}}^{(t)}
=
\max_{x_i\in\mathcal D_t}f(x_i),
$$

where $\mathcal D_t$ is the set of evaluated points available at iteration
$t$.

Because evaluations are retained,

$$
\mathcal D_t
\subseteq
\mathcal D_{t+1},
$$

and therefore

$$
f_{\mathrm{best}}^{(t+1)}
\geq
f_{\mathrm{best}}^{(t)}.
$$

Thus, the best-found objective value cannot decrease as the algorithm
progresses.

### Condition 8: Deterministic Evaluation

The function evaluation is deterministic:

$$
f(x)
$$

returns the same value whenever the same point $x$ is evaluated.

Therefore, the information collected by ARRGO is reproducible and does not
require probabilistic convergence arguments.

### Condition 9: Valid Certified Bounds

For certified convergence and $\varepsilon$-optimality, ARRGO additionally
requires a valid upper-bounding mechanism.

In Certified Mode, if a valid Lipschitz constant $L$ is available,

$$
|f(x)-f(y)|
\leq
L|x-y|,
\qquad
\forall x,y\in\Omega,
$$

then the regional upper envelope satisfies

$$
f(x)\leq U_R(x),
\qquad
\forall x\in R.
$$

Consequently,

$$
\max_{x\in R}f(x)
\leq
P(R)
=
\max_{x\in R}U_R(x).
$$

These bounds provide the mathematical basis for determining whether the
remaining unexplored potential of the search space is sufficiently small.

### Combined Convergence Requirement

The central convergence structure of ARRGO can therefore be expressed as

$$
\boxed{
\text{Coverage}
+
\text{Valid Refinement}
+
\text{Information Refinement}
+
\text{Fair Global Selection}
}
$$

together with continuity of the objective function.

For certified convergence, valid global bounds are additionally required:

$$
\boxed{
\text{Convergence Structure}
+
\text{Valid Bounds}
\Longrightarrow
\text{Certified Accuracy}
}
$$

These conditions separate the roles of the different components of ARRGO.

Spatial contraction controls geometric resolution.

Information refinement controls what the algorithm learns about each region.

Global selection controls where refinement is allocated.

Valid bounds provide the connection between finite observations and a
mathematical statement about the unexplored optimum.

The next step is to combine these conditions into a formal convergence
statement.

## Convergence Theorem

We now combine the convergence conditions into a formal statement about the
spatial behavior of ARRGO.

### Theorem: Spatial Convergence of Relevant Refinement

Let

$$
\Omega=[a,b]
$$

be a compact search domain, and let

$$
f\in C(\Omega).
$$

Assume that ARRGO satisfies the following conditions:

1. The collection of regions maintained by ARRGO preserves coverage of
   $\Omega$.

2. Every structural refinement satisfies the contraction condition

   $$
   \max\{\operatorname{diam}(R_L),
   \operatorname{diam}(R_R)\}
   \leq
   \rho\,\operatorname{diam}(R),
   \qquad
   0<\rho<1.
   $$

3. A region that remains globally relevant cannot be permanently ignored by
   the global selection mechanism.

4. Relevant regions continue to receive valid refinement opportunities.

Then, for every nested sequence of relevant regions

$$
R_0\supseteq R_1\supseteq R_2\supseteq\cdots,
$$

generated through repeated structural refinement,

$$
\operatorname{diam}(R_k)
\leq
\rho^k\operatorname{diam}(R_0).
$$

Consequently,

$$
\boxed{
\lim_{k\to\infty}\operatorname{diam}(R_k)=0
}
$$

### Proof

By the contraction condition, each refinement satisfies

$$
\operatorname{diam}(R_{k+1})
\leq
\rho\operatorname{diam}(R_k).
$$

Applying this inequality recursively gives

$$
\operatorname{diam}(R_k)
\leq
\rho^k\operatorname{diam}(R_0).
$$

Because

$$
0<\rho<1,
$$

we have

$$
\lim_{k\to\infty}\rho^k=0.
$$

Therefore,

$$
\lim_{k\to\infty}
\operatorname{diam}(R_k)
=
0.
$$

Hence, repeated valid refinement of a relevant nested region produces
arbitrarily fine spatial resolution.

### Consequence for the Objective Function

Since

$$
f\in C(\Omega),
$$

continuity implies that sufficiently small spatial neighborhoods around a
point produce arbitrarily small changes in the objective value.

For every $\eta>0$ and every $x\in\Omega$, there exists
$\delta>0$ such that

$$
|x-y|<\delta
\quad\Longrightarrow\quad
|f(x)-f(y)|<\eta.
$$

Since

$$
\operatorname{diam}(R_k)\to0,
$$

the spatial resolution of the relevant region can eventually become smaller
than any prescribed $\delta$.

Therefore, along a repeatedly refined relevant region, the possible
variation of the continuous objective becomes arbitrarily localized.

### Important Interpretation

This theorem establishes **spatial convergence of the refinement process**.

It does not, by itself, establish that the incumbent solution converges to a
global optimizer.

In particular,

$$
\operatorname{diam}(R_k)\to0
$$

does not imply

$$
x_{\mathrm{best}}^{(k)}\to x^*.
$$

To establish convergence of the optimization result, we additionally need
to connect:

$$
\boxed{
\text{Relevant Region Refinement}
\rightarrow
\text{Information Acquisition}
\rightarrow
\text{Global Optimization}
}
$$

This distinction is essential because an algorithm can refine regions
geometrically without necessarily evaluating sufficiently informative points
inside them.

### Certified Extension

If, in addition, $f$ satisfies a valid Lipschitz condition

$$
|f(x)-f(y)|
\leq
L|x-y|,
$$

then for any region $R_k$,

$$
\max_{x,y\in R_k}|f(x)-f(y)|
\leq
L\operatorname{diam}(R_k).
$$

Since

$$
\operatorname{diam}(R_k)\to0,
$$

we obtain

$$
L\operatorname{diam}(R_k)\to0.
$$

Thus, under the Lipschitz assumption, spatial contraction directly implies
that the maximum possible objective variation inside the repeatedly refined
region converges to zero.

This provides the mathematical bridge between spatial refinement and
certified uncertainty reduction.

The stronger statement that the global incumbent becomes
$\varepsilon$-optimal requires the global selection, bound validity, and
termination arguments established separately above.

## Global Convergence

Spatial convergence of a single nested region is not sufficient to establish
global convergence.

ARRGO is a global optimization framework, so its convergence mechanism must
also account for the entire collection of regions representing the search
domain.

Let

$$
\mathcal R_t
$$

denote the collection of regions maintained by ARRGO at iteration $t$.

The regions satisfy the coverage condition

$$
\Omega
\subseteq
\bigcup_{R\in\mathcal R_t}R.
$$

Therefore, every point in the original search domain remains represented by
at least one region.

### Global Relevance

Let

$$
R^*
$$

be a region that contains at least one global optimizer:

$$
R^*\cap X^*\neq\varnothing,
$$

where

$$
X^*
=
\operatorname*{arg\,max}_{x\in\Omega}f(x).
$$

For ARRGO to converge globally, the refinement mechanism must not permanently
ignore $R^*$ while refining other regions.

This is the role of the global selection fairness condition.

### Fair Global Selection

Suppose that a region containing a global optimizer remains globally
relevant according to the information maintained by ARRGO.

Then the selection mechanism must provide infinitely many refinement
opportunities to that region unless its information becomes sufficient to
resolve its relevance.

In other words, there cannot exist a permanently relevant region $R^*$ and a
finite iteration $T$ such that

$$
R^*
\text{ receives no further refinement after }T.
$$

This prevents permanent starvation of potentially optimal regions.

### Global Refinement Structure

The global convergence mechanism can therefore be represented as

$$
\boxed{
\text{Coverage}
+
\text{Fair Selection}
+
\text{Contraction}
}
$$

where:

- **Coverage** ensures that no part of $\Omega$ disappears from the global
  representation.
- **Fair Selection** ensures that globally relevant regions cannot be ignored
  forever.
- **Contraction** ensures that repeatedly refined regions obtain arbitrarily
  fine spatial resolution.

Together, these properties prevent the algorithm from permanently focusing on
an isolated portion of the search domain without giving relevant competing
regions refinement opportunities.

### Relation to a Global Optimizer

Consider a global optimizer

$$
x^*\in X^*.
$$

At every iteration, because the region collection preserves coverage, there
exists at least one region

$$
R_t^*
$$

such that

$$
x^*\in R_t^*.
$$

If this region remains globally relevant and continues to receive refinement,
then along a corresponding nested sequence

$$
R_0^*
\supseteq
R_1^*
\supseteq
R_2^*
\supseteq
\cdots
$$

we obtain

$$
\operatorname{diam}(R_t^*)\to0.
$$

Therefore, the spatial representation of a globally optimal location can be
made arbitrarily precise.

### From Spatial Convergence to Solution Convergence

The remaining question is whether this increasingly precise representation
causes ARRGO to obtain increasingly good objective values.

Continuity gives

$$
x_t\to x^*
\quad\Longrightarrow\quad
f(x_t)\to f(x^*).
$$

Thus, if ARRGO generates evaluated points

$$
x_t\in R_t^*
$$

that converge to a global optimizer,

$$
x_t\to x^*,
$$

then

$$
f(x_t)\to f(x^*)=f^*.
$$

Since the incumbent satisfies

$$
f_{\mathrm{best}}^{(t)}
\geq
f(x_t),
$$

it follows that the sequence of best-found objective values approaches the
global optimum from below:

$$
f_{\mathrm{best}}^{(t)}
\to
f^*.
$$

### Important Additional Requirement

The previous conclusion requires more than simply refining $R_t^*$.

The algorithm must actually generate evaluated points whose distance from the
relevant optimizer tends to zero.

Therefore, a complete convergence proof requires an additional
**evaluation-density or information-acquisition condition**.

A sufficient form is:

$$
\forall x^*\in X^*,
\qquad
\exists\{x_t\}
\text{ evaluated by ARRGO such that }
x_t\to x^*.
$$

Under this condition and continuity of $f$,

$$
f(x_t)\to f^*.
$$

Because the incumbent always retains the best evaluated value,

$$
f_{\mathrm{best}}^{(t)}
\geq
f(x_t),
$$

and because

$$
f_{\mathrm{best}}^{(t)}
\leq
f^*,
$$

we obtain

$$
\boxed{
f_{\mathrm{best}}^{(t)}
\to
f^*
}
$$

as the number of refinement opportunities tends to infinity.

### Global Convergence Principle

The global convergence logic of ARRGO can therefore be summarized as

$$
\boxed{
\begin{aligned}
&\text{Global Coverage}
\\
&\Downarrow
\\
&\text{Fair Selection of Relevant Regions}
\\
&\Downarrow
\\
&\text{Repeated Spatial Contraction}
\\
&\Downarrow
\\
&\text{Increasingly Precise Representation}
\\
&\Downarrow
\\
&\text{Evaluation Near Global Optimizers}
\\
&\Downarrow
\\
&f_{\mathrm{best}}^{(t)}\to f^*
\end{aligned}
}
$$

This establishes the conceptual structure required for global convergence.

A fully formal convergence theorem must additionally specify the exact
candidate-generation and action-selection rules that guarantee the required
evaluation-density condition.

Therefore, the implementation of ARRGO must preserve these theoretical
properties rather than relying only on heuristic region selection.

## Evaluation Density Condition

Spatial refinement guarantees that relevant regions can become arbitrarily
small. However, shrinking a region does not by itself guarantee that ARRGO
evaluates informative points sufficiently close to a global optimizer.

Therefore, the convergence analysis requires an additional condition on the
locations of function evaluations.

### Definition

Let

$$
\mathcal D_\infty
$$

denote the set of all points evaluated by ARRGO during an unbounded sequence
of iterations.

The evaluation set is said to satisfy the **global evaluation-density
condition** with respect to the global optimizer set $X^*$ if

$$
\forall x^*\in X^*,
\qquad
\inf_{x\in\mathcal D_\infty}|x-x^*|=0.
$$

Equivalently, for every global optimizer $x^*$ and every $\delta>0$, there
exists an evaluated point $x\in\mathcal D_\infty$ such that

$$
|x-x^*|<\delta.
$$

Thus, evaluated points occur arbitrarily close to every global optimizer.

### Why This Condition Is Necessary

Suppose ARRGO repeatedly refines a region containing a global optimizer
$x^*$.

It is possible, in principle, for the algorithm to reduce the diameter of
that region while repeatedly evaluating points that remain away from $x^*$.

Therefore,

$$
\operatorname{diam}(R_t)\to0
$$

alone does not imply

$$
\exists\,x_t\in\mathcal D_\infty:
x_t\to x^*.
$$

The evaluation-density condition explicitly rules out this failure mode.

### Connection with Continuity

Let

$$
x^*\in X^*
$$

and suppose the evaluation-density condition holds.

Then there exists a sequence of evaluated points

$$
x_k\in\mathcal D_\infty
$$

such that

$$
x_k\to x^*.
$$

Since

$$
f\in C(\Omega),
$$

continuity gives

$$
f(x_k)\to f(x^*).
$$

Because $x^*$ is a global optimizer,

$$
f(x^*)=f^*.
$$

Therefore,

$$
\boxed{
f(x_k)\to f^*
}
$$

as the evaluated points approach the global optimizer.

### Consequence for the Incumbent

The ARRGO incumbent is defined by

$$
f_{\mathrm{best}}^{(t)}
=
\max_{x\in\mathcal D_t}f(x).
$$

Since every evaluated point is retained,

$$
\mathcal D_t
\subseteq
\mathcal D_{t+1},
$$

and therefore

$$
f_{\mathrm{best}}^{(t+1)}
\geq
f_{\mathrm{best}}^{(t)}.
$$

For the sequence $x_k\to x^*$ described above,

$$
f(x_k)\to f^*.
$$

Since the incumbent is at least as good as every previously evaluated
point,

$$
f_{\mathrm{best}}^{(t_k)}
\geq
f(x_k)
$$

at the corresponding iterations $t_k$.

At the same time,

$$
f_{\mathrm{best}}^{(t)}
\leq
f^*
$$

for every iteration.

Hence,

$$
\boxed{
\lim_{t\to\infty}f_{\mathrm{best}}^{(t)}
=
f^*
}
$$

provided that the global evaluation-density condition holds.

### Relation to Region Refinement

The evaluation-density condition can be achieved through the interaction of
three mechanisms:

$$
\boxed{
\text{Fair Region Selection}
+
\text{Spatial Contraction}
+
\text{Informative Evaluation}
}
$$

Fair selection prevents a globally relevant region from being permanently
ignored.

Spatial contraction makes the region containing a global optimizer
arbitrarily small.

Informative evaluation ensures that function evaluations are actually
generated sufficiently close to the optimizer.

### Certified Interpretation

In Certified Mode, the evaluation-density condition is not the only route to
a practical stopping guarantee.

If ARRGO obtains a valid global upper bound satisfying

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}-f_{\mathrm{best}}
\leq
\varepsilon,
$$

then $\varepsilon$-optimality follows directly from the certified gap,
regardless of whether the exact optimizer location has been identified.

Thus, two different theoretical goals must be distinguished:

$$
\boxed{
\text{Asymptotic Convergence}
}
$$

requires increasingly informative evaluations near global optimizers, whereas

$$
\boxed{
\text{Finite-Time Certified Accuracy}
}
$$

can be established through a sufficiently small valid global optimality gap.

### Design Requirement for ARRGO

The implementation must therefore ensure that its candidate-generation and
selection mechanisms are compatible with the evaluation-density condition.

The condition should not be assumed merely because regions are repeatedly
split.

It must emerge from the actual refinement policy used by ARRGO.

This requirement will later be used when formalizing the exact sampling and
splitting rules.